# KAVACH — teach Llama 3.2 1B our scam playbook (LoRA) and make the phone file

**What this does:** downloads Llama 3.2 1B Instruct, trains a small add-on (LoRA) on KAVACH's
synthetic scam-call examples, checks the answers before and after, and converts the result to the
ExecuTorch `.pte` file the phone app loads. Everything is saved to your Google Drive under `kavach/`.

**Before you start**
1. Hugging Face account → open <https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct> → accept the licence (approval is usually minutes).
2. <https://huggingface.co/settings/tokens> → *Create new token* → type **Read** → copy it.
3. In Colab: **Runtime → Change runtime type → T4 GPU → Save**. Then **Runtime → Run all**.

You will be asked twice: allow Google Drive, and paste the token. Total time ≈ 1.5–2 h on a free T4.
Nothing here uses real people's calls; the data is generated by `data/synth/generate_calls.py`.
The locked held-out test split is **never** used for training or tuning.

In [1]:
# 1) Check the GPU and connect Google Drive (results are copied there at each stage)
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU: Runtime > Change runtime type > T4 GPU"
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/kavach'
os.makedirs(OUT, exist_ok=True)
print('Results folder:', OUT)

name, memory.total [MiB]
Tesla T4, 15360 MiB
Mounted at /content/drive
Results folder: /content/drive/MyDrive/kavach


In [2]:
# 2) Get the KAVACH code and the training libraries
%cd /content
!rm -rf KAVACH && git clone -q --depth 1 https://github.com/siddhaantxsingh/KAVACH.git
%cd /content/KAVACH
!pip -q install -r models/llama/requirements-train.txt
!git log --oneline -1

/content
/content/KAVACH
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
b3be4fa (grafted, HEAD -> main, origin/main, origin/HEAD) Rebuild report PDF (Phase 2 plan mentions the Colab notebook)


In [3]:
# 3) Hugging Face token (paste it; it is not printed or saved in the notebook)
from getpass import getpass
from huggingface_hub import login, hf_hub_download
tok = getpass('Paste your Hugging Face READ token: ').strip()
login(token=tok)
os.environ['HF_TOKEN'] = tok
BASE = 'meta-llama/Llama-3.2-1B-Instruct'
# Fails here with 403 if the licence was not accepted yet on the model page.
TOKENIZER = hf_hub_download(BASE, 'original/tokenizer.model')
print('Access OK. Tokenizer:', TOKENIZER)

Paste your Hugging Face READ token: ··········


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


original/tokenizer.model:   0%|          | 0.00/2.18M [00:00<?, ?B/s]

Access OK. Tokenizer: /root/.cache/huggingface/hub/models--meta-llama--Llama-3.2-1B-Instruct/snapshots/9213176726f574b556790deb65791e0c5aa438b6/original/tokenizer.model


In [4]:
# 4) Make the training examples (synthetic calls -> question/answer pairs in the phone's exact prompt format)
!python data/synth/generate_calls.py
!python models/llama/make_sft_data.py
!wc -l data/synth/out/sft_train.jsonl data/synth/out/sft_dev.jsonl

train: 8000 calls (3982 scam, 4018 benign) -> /content/KAVACH/data/synth/out/calls_train.jsonl
dev: 600 calls (288 scam, 312 benign) -> /content/KAVACH/data/synth/out/calls_dev.jsonl
test: 1000 calls (511 scam, 489 benign) -> /content/KAVACH/data/synth/out/calls_test.jsonl
test_far: 1000 calls (502 scam, 498 benign) -> /content/KAVACH/data/synth/out/calls_test_far.jsonl
train 4000 scam share 0.483
dev 400 scam share 0.448
    4000 data/synth/out/sft_train.jsonl
     400 data/synth/out/sft_dev.jsonl
    4400 total


In [5]:
# 5) Score the ORIGINAL model first (so the report can show before vs after). ~8 min.
N_EVAL = 100
!python models/llama/eval_llama.py --hf {BASE} --n {N_EVAL} --name llama_base
!cp bench/results/llama_base.json {OUT}/

tokenizer_config.json: 54.5kB [00:00, 130MB/s]
tokenizer.json: 9.09MB [00:00, 27.9MB/s]
special_tokens_map.json: 100% 296/296 [00:00<00:00, 2.81MB/s]
config.json: 100% 877/877 [00:00<00:00, 7.52MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1794415363.878694    2079 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794415366.308819    2079 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
model.safetensors: 100% 2.47G/2.47G [00:29<00:00, 82.8MB/s]
generation_config.json: 100% 189/189 [00:00<00:00, 903kB/s]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from in

In [8]:
   !pip -q uninstall -y torchao

In [9]:
# 6) Train the LoRA add-on. ~45-75 min on a T4. Saves the small adapter and a merged copy.
!python models/llama/finetune_lora.py --base {BASE} \
    --train data/synth/out/sft_train.jsonl --dev data/synth/out/sft_dev.jsonl \
    --out models/llama/work/kavach-lora --epochs 1
!cp -r models/llama/work/kavach-lora/adapter {OUT}/lora_adapter
print('Adapter saved to Drive (small). If anything later fails, this is the part worth keeping.')

I0000 00:00:1794416285.155237    6181 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794416287.570981    6181 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
`torch_dtype` is deprecated! Use `dtype` instead!
trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039
4000 train / 400 dev examples after length filter
  0% 0/250 [00:00<?, ?it/s]`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
  0% 1/250 [00:49<3:23:26, 49.02s/it][W1111 16:59:57.292581486 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 1031798784 bytes (free: 893190144, total: 15637086208).
{'loss': 1.2762, 'grad_norm': 1.301885962486267, 'learni

In [10]:
# 7) Score the TRAINED model on the same examples. ~8 min.
!python models/llama/eval_llama.py --hf models/llama/work/kavach-lora/merged --n {N_EVAL} --name llama_lora
!cp bench/results/llama_lora.json {OUT}/
import json
b = json.load(open('bench/results/llama_base.json')); l = json.load(open('bench/results/llama_lora.json'))
print(f"{'metric':16s} {'before':>8s} {'after':>8s}")
for k in ['json_valid','scam_acc','type_acc','stop_stage_acc','evidence_ok','copied_example','say_lang_ok','sec_per_example']:
    print(f"{k:16s} {b.get(k,'-'):>8} {l.get(k,'-'):>8}")

The tokenizer you are loading from 'models/llama/work/kavach-lora/merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1794431314.605171   68928 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794431323.662690   68928 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not

In [11]:
# 8) Convert to the phone format. ExecuTorch MUST be 1.4.0 = the version inside the app.
#    (Installing it changes torch; that is fine because every step runs as a fresh `python` process.)
!pip -q install -r models/llama/requirements.txt
!python -c "import executorch, torch; print('executorch OK, torch', torch.__version__)"
W = 'models/llama/work'
!python models/llama/hf_to_meta.py {W}/kavach-lora/merged {W}/meta
!rm -rf {W}/kavach-lora/merged   # free disk/RAM before export
!python -m executorch.extension.llm.export.export_llm --config models/llama/configs/kavach_xnnpack_8da4w.yaml \
    +base.checkpoint={W}/meta/consolidated.00.pth +base.params={W}/meta/params.json \
    +export.output_name={W}/kavach_llama3_2_1b.pte
!ls -la {W}/*.pte

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
executorch OK, torch 2.14.1+cu130
wrote models/llama/work/meta/consolidated.00.pth (147 tensors) and params.json
W1111 21:16:31.526000 71025 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
I tokenizers:regex.cpp:40] Registering override fallback regex
[2026-11-11 21:16:36,533][root][INFO] - Applying quantizers: []
[2026-11-11 21:16:36,923][root][INFO] - Checkpoint dtype: torch.float32
quantize ('tok_embeddings', Embedding(128256, 2048)) with group_size 32, bitwidth 4
[2026-11-11 21:22:04,852][root][INFO] - Looking for libcustom_ops_aot_lib.so in /usr/local/lib/python3.13/dist-packages/executorch/extension/llm/custom_ops
[2026-11-11 21:22:04,856][root][INFO

In [12]:
# 9) Copy the phone file + tokenizer to Drive and write a short summary
import shutil, datetime
pte = 'models/llama/work/kavach_llama3_2_1b.pte'
shutil.copy(pte, f'{OUT}/kavach_llama3_2_1b.pte')
shutil.copy(TOKENIZER, f'{OUT}/tokenizer.model')
json.dump({'base': BASE, 'epochs': 1, 'n_eval': N_EVAL, 'before': b, 'after': l,
           'pte_mb': round(os.path.getsize(pte)/1e6, 1), 'executorch': '1.4.0',
           'made': datetime.datetime.utcnow().isoformat() + 'Z'},
          open(f'{OUT}/lora_summary.json', 'w'), indent=2)
print(open(f'{OUT}/lora_summary.json').read())
print('''
DONE. In Google Drive > kavach you now have:
  kavach_llama3_2_1b.pte   (phone file, about 1 GB)
  tokenizer.model
  llama_base.json, llama_lora.json, lora_summary.json   (send these to Claude for the report)

Put the phone file on the phone (Git Bash, phone plugged in, from the KAVACH folder):
  scripts/push_models.sh --llama ~/Downloads/kavach_llama3_2_1b.pte ~/Downloads/tokenizer.model
To go back to the official model:  scripts/push_models.sh --official-spinquant
''')

{
  "base": "meta-llama/Llama-3.2-1B-Instruct",
  "epochs": 1,
  "n_eval": 100,
  "before": {
    "name": "llama_base",
    "n": 100,
    "json_valid": 0.71,
    "scam_acc": 0.38,
    "type_acc": 0.5806,
    "stop_stage_acc": 0.7097,
    "evidence_ok": 0.0845,
    "copied_example": 0.3239,
    "say_lang_ok": 0.5493,
    "sec_per_example": 4.08
  },
  "after": {
    "name": "llama_lora",
    "n": 100,
    "json_valid": 1.0,
    "scam_acc": 0.89,
    "type_acc": 0.7556,
    "stop_stage_acc": 1.0,
    "evidence_ok": 0.96,
    "copied_example": 0.0,
    "say_lang_ok": 1.0,
    "sec_per_example": 3.35
  },
  "pte_mb": 860.4,
  "executorch": "1.4.0",
  "made": "2026-11-11T21:30:18.189242Z"
}

DONE. In Google Drive > kavach you now have:
  kavach_llama3_2_1b.pte   (phone file, about 1 GB)
  tokenizer.model
  llama_base.json, llama_lora.json, lora_summary.json   (send these to Claude for the report)

Put the phone file on the phone (Git Bash, phone plugged in, from the KAVACH folder):
  script

/tmp/ipykernel_430/1912445991.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'made': datetime.datetime.utcnow().isoformat() + 'Z'},


### If something goes wrong
* **403 / "gated repo"** in step 3 → the licence is not approved yet on the model page; wait, then re-run from step 3.
* **"CUDA out of memory"** in step 6 → Runtime → Restart session, then run steps 1–4 and 6 again with `--max-len 1024` added.
* **Session crashed / ran out of RAM** in step 8 → the adapter is already safe in Drive (`kavach/lora_adapter`). Run steps 1–3, then the last cell below (rebuilds the merged model from the adapter), then step 8 again.

In [13]:
# (Only if step 8 crashed) Rebuild the merged model from the adapter saved in Drive, then run step 8 again.
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM
m = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float32)
m = PeftModel.from_pretrained(m, f'{OUT}/lora_adapter').merge_and_unload()
m.save_pretrained('models/llama/work/kavach-lora/merged', safe_serialization=True)
print('merged model rebuilt')

W1111 21:30:51.733000 430 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1111 21:30:51.855000 430 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
`torch_dtype` is deprecated! Use `dtype` instead!


merged model rebuilt
